In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 설정된 폰트 목록 출력
sys_font = [f.name for f in fm.fontManager.ttflist]
print(sys_font)

# 나눔고딕 폰트가 목록에 있는지 확인
print('NanumGothic' in sys_font)

In [ ]:
!pip install koreanize-matplotlib

In [9]:
import koreanize_matplotlib

# 한국어 형태소 분석 및 데이터 시각화

이 노트북은 Bareun API를 사용하여 한국어 텍스트 데이터에서 명사를 추출하고, 다양한 시각화 기법을 통해 분석하는 방법을 설명합니다.

## 주요 기능
1. Bareun API를 사용한 한국어 형태소 분석
2. 뉴스 데이터에서 명사 추출
3. 워드클라우드, 네트워크 분석, 막대 차트를 통한 시각화

In [ ]:
# Bareun API 패키지 다운로드
# curl 명령을 사용해 Bareun Linux 패키지를 다운로드
# -L: 리다이렉션을 따름, -J: Content-Disposition 헤더 존재 시 파일명 사용, -k: SSL 인증 검증 무시, -s: 진행 상황 표시 없음
!curl -LJks -H "uname:$(uname -a)" https://bareun.ai/api/get -o bareun-linux.deb

# 현재 디렉토리의 파일 목록 표시
!ls

In [ ]:
# 시스템 정보 출력
# uname -a: 운영체제 이름, 호스트명, 커널 버전 등의 시스템 정보 출력
!uname -a

In [ ]:
# 다운로드한 Bareun 패키지 설치
# dpkg -i: 데비안 패키지 설치
!dpkg -i bareun-linux.deb

In [ ]:
# Bareun 환경 변수 설정
# %env: Jupyter 매직 명령어로 환경 변수 설정
%env BAREUN_ROOT="/opt/bareun"  # Bareun 설치 루트 경로 설정
%env LD_LIBRARY_PATH="/opt/bareun/lib"  # 라이브러리 경로 설정

# Bareun 서비스 백그라운드로 실행
# nohup: 로그아웃 후에도 명령이 계속 실행되도록 함
# &: 명령을 백그라운드로 실행
!BAREUN_ROOT="/opt/bareun" LD_LIBRARY_PATH="/opt/bareun/lib" nohup /opt/bareun/bin/bareun&

In [ ]:
# Bareun 프로세스 실행 확인
# ps -ef: 모든 프로세스 상세 정보 출력
# grep bareun: bareun 문자열이 포함된 라인만 필터링
!ps -ef | grep bareun

In [ ]:
# API 키 등록
# -reg 옵션: API 키 등록
!BAREUN_ROOT="/opt/bareun" LD_LIBRARY_PATH="/opt/bareun/lib" /opt/bareun/bin/bareun -reg koba-YEVHS7Q-VDSUWIY-XCIS3OQ-LWD7WHA

In [ ]:
# Bareun Python 패키지(bareunpy) 설치 또는 업데이트
# -U: 이미 설치된 패키지를 최신 버전으로 업그레이드
!pip install -U bareunpy

## 1단계: Bareun API 초기화

이제 Bareun API를 사용하기 위해 필요한 라이브러리를 임포트하고 API를 초기화합니다.

### 주요 컴포넌트:
- **Tagger**: 형태소 분석을 수행하는 객체
- **Tokenizer**: 토크나이징을 수행하는 객체
- **API_KEY**: Bareun API 사용을 위한 인증 키

In [17]:
# 필요한 라이브러리 임포트
import sys  # 시스템 관련 함수 및 변수 제공
import google.protobuf.text_format as tf  # 프로토콜 버퍼 텍스트 포맷 처리
from bareunpy import Tagger  # 형태소 분석기
from bareunpy import Tokenizer  # 토크나이저
from collections import defaultdict  # 기본값이 있는 딕셔너리

# Bareun API 초기화
API_KEY="koba-YEVHS7Q-VDSUWIY-XCIS3OQ-LWD7WHA"  # API 키 설정
tagger = Tagger(API_KEY, 'localhost', 5656)  # 형태소 분석기 객체 생성 (localhost:5656 서버에 연결)
tokenizer = Tokenizer(API_KEY, 'localhost', 5656)  # 토크나이저 객체 생성

In [ ]:
# 연습: 위의 코드를 따라서 타이핑해보세요


## 2단계: 데이터 불러오기

분석할 뉴스 데이터를 Excel 파일에서 불러옵니다.

### 주요 포인트:
- pandas를 사용하여 Excel 파일을 DataFrame으로 로드
- 데이터의 구조와 내용을 확인
- 컬럼명과 데이터 타입 파악

In [ ]:
# pandas 라이브러리 임포트 (데이터 처리)
import pandas as pd

# Excel 파일 불러오기
file_path = '교권_news.xlsx'  # 분석할 Excel 파일 경로

# DataFrame으로 Excel 파일 로드
# sheet_name=0: 첫 번째 시트 선택
df = pd.read_excel(file_path, sheet_name=0)

# 데이터 미리보기 출력
print("Excel file loaded successfully. Preview:")  # 성공 메시지
print(df.head())  # 처음 5개 행 출력

# 데이터프레임 기본 정보 출력
print("\nDataFrame info:")
print(f"Shape: {df.shape}")  # 행과 열의 수 (shape)
print(f"Columns: {df.columns.tolist()}")  # 컬럼명 목록

In [ ]:
# 연습: 데이터 불러오기 코드를 작성해보세요


## 3단계: 명사 추출 및 빈도 분석

이제 본격적으로 형태소 분석을 수행합니다.

### 작업 과정:
1. 제목과 본문에서 명사만 추출하는 함수 정의
2. 한 글자 명사는 제외 (의미가 모호하기 때문)
3. 모든 명사의 빈도수를 계산
4. 상위 20개 명사 추출

### 형태소 분석의 이해:
- 형태소: 의미를 가진 언어의 최소 단위
- 명사 추출: 텍스트의 핵심 키워드를 파악하는 데 유용
- 빈도 분석: 어떤 주제가 가장 많이 언급되는지 파악

In [ ]:
# 제목/본문에서 명사 추출 함수 (한 글자 명사 제외)
def extract_nouns_from_texts(texts, tagger):
    all_nouns = []
    for text in texts:
        if not isinstance(text, str):
            continue
        nouns = tagger.tags([text]).nouns()
        nouns = [noun for noun in nouns if len(noun) > 1]  # 한 글자 명사 제외
        all_nouns.extend(nouns)
    return all_nouns

In [ ]:
# 연습: 명사 추출 함수를 작성해보세요


In [ ]:
# 제목에서 명사 추출
title_texts = df['제목'].dropna().tolist()
title_nouns = extract_nouns_from_texts(title_texts, tagger)

# 본문에서 명사 추출
content_texts = df['본문'].dropna().tolist()
content_nouns = extract_nouns_from_texts(content_texts, tagger)

# 제목+본문 전체 명사 합치기
all_nouns = title_nouns + content_nouns

# 빈도수 집계
from collections import Counter
noun_counts = Counter(all_nouns)
top_nouns = noun_counts.most_common(20)

print("상위 20개 명사와 빈도:")
for noun, count in top_nouns:
    print(f"{noun}: {count}")

In [ ]:
# 연습: 명사 추출 및 빈도 계산 코드를 작성해보세요


## 4단계: 워드클라우드 시각화

추출된 명사들을 워드클라우드로 시각화합니다.

### 워드클라우드란?
- 텍스트 데이터에서 중요한 단어들을 시각적으로 표현
- 단어의 빈도에 따라 글자 크기가 결정됨
- 직관적으로 주요 키워드를 파악할 수 있음

### 한글 처리 주의사항:
- 한글 폰트 지정 필수 (NanumGothic.ttf)
- 폰트 경로를 정확히 설정해야 함

In [ ]:
# 워드클라우드 시각화

from wordcloud import WordCloud  # 워드클라우드 생성을 위한 라이브러리 임포트
import matplotlib.pyplot as plt  # 그래프(이미지) 출력을 위한 라이브러리 임포트

# 워드클라우드 객체 생성
# - font_path: 한글 폰트 경로 지정 (한글 깨짐 방지)
# - width, height: 워드클라우드 이미지 크기 지정
# - background_color: 배경색 지정
wordcloud = WordCloud(font_path='NanumGothic.ttf', width=800, height=400, background_color='white')

# top_nouns(상위 20개 명사와 빈도) 정보를 워드클라우드에 적용
# dict(top_nouns): [('단어', 빈도), ...] 형태를 딕셔너리로 변환
wordcloud.generate_from_frequencies(dict(top_nouns))

# 워드클라우드 이미지를 그릴 도화지(figure) 생성, 크기 지정
plt.figure(figsize=(10, 5))

# 워드클라우드 이미지를 화면에 표시
# interpolation='bilinear'는 이미지를 부드럽게 보이게 함
plt.imshow(wordcloud, interpolation='bilinear')

# x, y축 눈금(테두리) 숨기기
plt.axis('off')

# 실제로 이미지를 화면에 출력
plt.show()

In [ ]:
# 연습: 워드클라우드 생성 코드를 작성해보세요


## 5단계: 막대그래프 시각화

명사 빈도를 막대그래프로 시각화합니다.

### 막대그래프의 장점:
- 정확한 수치 비교가 가능
- 순위를 명확하게 파악할 수 있음
- 워드클라우드보다 정량적 분석에 유리

### 시각화 팁:
- x축 라벨을 45도 회전하여 가독성 향상
- 적절한 그래프 크기 설정으로 보기 좋게 구성

In [ ]:
# 막대그래프 시각화
import matplotlib.pyplot as plt  # 그래프를 그리기 위한 라이브러리 임포트

# top_nouns는 [('단어', 빈도), ...] 형태의 리스트임
# zip(*top_nouns)를 사용하면 단어와 빈도수를 각각 분리해서 labels, values에 저장
labels, values = zip(*top_nouns)

# 그래프의 크기를 가로 10, 세로 5로 설정
plt.figure(figsize=(10, 5))

# 막대그래프(bar chart) 그리기: labels(단어)가 x축, values(빈도수)가 y축
plt.bar(labels, values)

# x축의 단어들이 겹치지 않도록 45도 기울여서 표시
plt.xticks(rotation=45)

# 그래프의 제목 설정
plt.title('상위 20개 명사 빈도')

# 그래프를 화면에 출력
plt.show()

In [ ]:
# 연습: 막대그래프 생성 코드를 작성해보세요


# 🌐 6단계: 교권 뉴스 단어 네트워크 분석

## 💡 학습 목표
- 네트워크란 무엇인가?
- 뉴스 데이터에서 핵심 키워드 찾기
- 단어들 간의 관계 시각화하기

---
## 📍 셀 1: 초기 설정 (그냥 "실행"만 하세요!)

⚠️ **주의:** 이 셀을 먼저 실행해야 합니다. 오류가 나면 알려주세요.

In [ ]:
# ========== 수업 전 미리 준비 ==========
# (학생들은 이 셀을 그냥 "실행"만 하면 됨)

import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter, defaultdict
import koreanize_matplotlib
from bareunpy import Tagger
import pandas as pd

# 한글 폰트 설정
plt.rcParams['font.family'] = 'NanumGothic'

# Bareun 초기화
API_KEY = "koba-YEVHS7Q-VDSUWIY-XCIS3OQ-LWD7WHA"
tagger = Tagger(API_KEY, 'localhost', 5656)

# 데이터 로드
df = pd.read_excel('교권_news.xlsx', sheet_name=0)

print("✅ 준비 완료! 모든 데이터가 로드되었습니다.")
print(f"📰 뉴스 개수: {len(df)}개")
print(f"📋 컬럼: {list(df.columns)}")

---
## 📍 셀 2: 도움말 함수들 (미리 만들어진 함수)

⚠️ **주의:** 이 함수들의 안에 있는 코드는 복잡하지만, 그냥 "사용"만 하면 됩니다.

In [ ]:
# 📍 함수 1: 상위 단어 추출
def get_top_words(df, num=15):
    """데이터에서 상위 단어 추출"""
    # 모든 명사를 저장할 빈 리스트 생성
    all_nouns = []
    
    # 데이터프레임의 각 행을 반복 순회
    for idx, row in df.iterrows():
        # 제목과 본문을 공백으로 연결하여 하나의 텍스트로 만듦
        text = str(row['제목']) + " " + str(row['본문'])
        # Bareun 형태소 분석기를 사용하여 명사만 추출
        nouns = tagger.tags([text]).nouns()
        # 한 글자 명사는 제외하고 1글자 초과 명사만 리스트에 추가
        all_nouns.extend([n for n in nouns if len(n) > 1])
    
    # Counter를 사용하여 명사의 빈도수를 계산하고 상위 num개를 딕셔너리로 변환하여 반환
    return dict(Counter(all_nouns).most_common(num))


# 📍 함수 2: 네트워크 생성
def create_word_network(df, top_n=15, min_cooccur=2):
    """단어 네트워크 자동 생성"""
    # 각 문서의 명사들을 저장할 리스트 생성
    doc_nouns = []
    
    # 데이터프레임의 각 행을 반복 순회
    for idx, row in df.iterrows():
        # 제목과 본문을 공백으로 연결
        text = str(row['제목']) + " " + str(row['본문'])
        # 형태소 분석 후 한 글자 초과 명사만 필터링하여 리스트에 저장
        nouns = [n for n in tagger.tags([text]).nouns() if len(n) > 1]
        doc_nouns.append(nouns)
    
    # doc_nouns의 모든 리스트에서 명사를 하나의 리스트로 평탄화
    all_nouns = [n for nouns in doc_nouns for n in nouns]
    # Counter를 사용하여 빈도수가 높은 상위 top_n개 명사를 집합으로 변환
    top_words = set([n for n, _ in Counter(all_nouns).most_common(top_n)])
    # 모든 명사의 빈도수 계산
    noun_counts = Counter(all_nouns)
    
    # 단어 공존(함께 나타남) 빈도를 저장할 기본값이 0인 딕셔너리 생성
    co_occur = defaultdict(int)
    # 각 문서의 명사들에 대해 반복
    for nouns in doc_nouns:
        # 상위 top_n개 단어에만 포함된 명사들만 필터링
        filtered = [n for n in nouns if n in top_words]
        # 필터링된 명사들의 모든 쌍(조합)에 대해 반복
        for i in range(len(filtered)):
            for j in range(i+1, len(filtered)):
                # 두 단어를 정렬하여 순서 상관없이 동일한 쌍으로 만듦
                pair = tuple(sorted([filtered[i], filtered[j]]))
                # 해당 쌍의 공존 빈도 증가
                co_occur[pair] += 1
    
    # 네트워크 그래프 객체 생성 (무방향 그래프)
    G = nx.Graph()
    # 공존 빈도가 최소 기준(min_cooccur) 이상인 쌍만 그래프에 추가
    for (a, b), cnt in co_occur.items():
        if cnt >= min_cooccur:
            # a와 b 노드를 연결하고 가중치(cnt)를 간선에 할당
            G.add_edge(a, b, weight=cnt)
    
    # 생성된 네트워크 그래프와 명사 빈도수 딕셔너리를 반환
    return G, noun_counts


# 모든 함수가 정상적으로 정의되었음을 알리는 메시지 출력
print("✅ 함수 준비 완료!")

---
## 📍 셀 3: 상위 단어 확인

### 👇 아래 코드를 실행해보세요!

**질문들:**
- ❓ 가장 자주 나오는 단어는?
- ❓ 교권과 관련해서 어떤 단어들이 자주 나올까?

In [ ]:
# 상위 15개 단어 추출
top_words = get_top_words(df, num=15)

print("📊 교권 뉴스에서 가장 많이 나온 단어 TOP 15:")
print("=" * 50)
for i, (word, freq) in enumerate(top_words.items(), 1):
    bar = "■" * (freq // 10)  # 시각적 표현
    print(f"{i:2d}. {word:8s} {bar} {freq}회")

print("\n💡 해석:")
print("   '교사'와 '교권'이 가장 많이 나온다")
print("   → 이들이 교권 이슈의 핵심 키워드!")

---
## 📍 셀 4: 네트워크 그리기

### 👇 아래 코드를 실행해보세요!

**질문들:**
- ❓ 어떤 단어가 중심에 있나요?
- ❓ "교사"와 "교권"이 왜 연결되어 있나요?
- ❓ "경찰"은 왜 따로 떨어져 있나요?

In [ ]:
# 네트워크 생성
G, word_freq = create_word_network(df, top_n=15, min_cooccur=2)

print(f"✅ 네트워크 생성 완료!")
print(f"   🔹 단어: {G.number_of_nodes()}개")
print(f"   🔹 연결: {G.number_of_edges()}개")

# 네트워크 그리기
plt.figure(figsize=(12, 9))

# 📍 단어 배치 (중요한 단어가 중심으로)
pos = nx.spring_layout(G, seed=42, k=3.0, iterations=100)

# 📍 단어 크기 (많이 나온 단어일수록 크게)
node_sizes = [word_freq[node] * 8 for node in G.nodes()]

# 단어 그리기
nx.draw_networkx_nodes(
    G, pos, 
    node_size=node_sizes,
    node_color='lightblue', 
    edgecolors='navy', 
    linewidths=2.5
)

# 연결선 그리기
nx.draw_networkx_edges(G, pos, alpha=0.25, edge_color='gray')

# 단어 이름 표시
nx.draw_networkx_labels(G, pos, font_family='NanumGothic', font_size=12)

plt.title("교권 뉴스 단어 네트워크\n(선 = 같은 뉴스에 함께 나타남)", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

print("\n💡 해석:")
print("   → '교사'와 '교권'이 중심에 있다")
print("   → '경찰', '고소'는 따로 묶여있다")

---
## 📍 셀 5: 네트워크 통계 분석

### 👇 아래 코드를 실행해보세요!

**"가장 중요한 단어"를 2가지 방법으로 찾아봅시다!**

In [ ]:
print("🔍 교권 뉴스의 핵심 단어는?\n")
print("=" * 60)

# 방법 1: 빈도순
print("\n1️⃣ 가장 많이 나온 단어 TOP 5:")
top_5_words = list(word_freq.most_common(5))
for word, freq in top_5_words:
    bar = "█" * (freq // 15)
    print(f"   {word:8s} {bar} {freq}회")

# 방법 2: 연결순
print("\n2️⃣ 가장 많이 연결된 단어 TOP 5:")
node_degrees = dict(G.degree())
top_5_connected = sorted(node_degrees.items(), key=lambda x: x[1], reverse=True)[:5]
for word, degree in top_5_connected:
    connections = "◆" * degree
    print(f"   {word:8s} {connections} ({degree}개와 연결)")

# 📊 비교 차트 추가
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 빈도수 차트
words1, freqs = zip(*top_5_words)
ax1.barh(words1, freqs, color='skyblue')
ax1.set_xlabel('빈도수')
ax1.set_title('📊 많이 나온 단어')

# 연결수 차트
words2, degrees = zip(*top_5_connected)
ax2.barh(words2, degrees, color='salmon')
ax2.set_xlabel('연결 개수')
ax2.set_title('🔗 많이 연결된 단어')

plt.tight_layout()
plt.show()

print("\n💡 해석:")
print("   두 차트 모두 '교사'와 '교권'이 상위권!")
print("   → 이들이 교권 이슈의 핵심!")

---
## 🎯 결론

### ✅ 네트워크 분석이란?
- 단어들이 어떻게 연결되어 있는지 보는 것
- 뉴스의 주요 주제와 구조를 파악 가능

### ✅ 이 분석으로 알 수 있는 것
- **"교사"와 "교권"** 이 교권 이슈의 중심 키워드
- **법적 처리**(경찰, 고소)는 별도 관점으로 다루어짐
- **교권 이슈**가 여러 측면에서 복합적으로 논의됨

### ✅ 비판적 질문
❓ 이 네트워크에 보이지 않는 것은?
- 학생들의 학습권, 교사들의 심리 상담, 부모들의 진정한 의견 등

❓ 왜 특정 관점만 보도되는가?
- 언론은 뉴스 가치가 높은 것부터 보도
- 특정 주제에 더 집중

❓ 이것이 중요한 이유?
- **미디어 리터러시** - 뉴스를 비판적으로 읽는 능력
- 보이는 것뿐만 아니라 보이지 않는 것을 생각하기

---
## 📚 생각해볼 점

### 🤔 추가 질문들

**Q1: "제주"가 왜 많이 나올까?**
- A: 제주 지역의 교권 침해 사건 뉴스가 많아서
- → 특정 지역 이슈가 언론에서 자주 다루어짐

**Q2: "경찰", "고소", "수사"가 왜 함께 떨어져 있을까?**
- A: 법적 처리 과정을 다루는 뉴스들
- → 교권 문제를 법적 관점에서 다룬 기사들

**Q3: "학생", "학부모"는?**
- A: 교권 이슈와 관련된 다른 주체들
- → 뉴스에서 어떻게 표현되는지 주목

**Q4: 만약 6개월 뒤에 같은 분석을 하면?**
- A: 같은 단어들이 나오지만 연결이 바뀔 수 있음
- → 시간에 따라 이슈의 초점이 변함

---

### 🎓 마지막 메시지

**뉴스는 사실을 전달하지만, 특정 관점에서 선택된 사실입니다.**

이 네트워크 분석을 통해:
- ✅ 뉴스의 주요 키워드 파악
- ✅ 뉴스의 각도와 관점 이해
- ✅ 비판적 사고 능력 발전
- ✅ 미디어 리터러시 향상